# Evaluate Responses With Gemini API

Notebook này đọc `instruction-data-with-response.json` có cấu trúc:

```json
{
  "instruction": "...",
  "input": "...",
  "output": "...",
  "model_response": "..."
}
```

Sau đó gọi Gemini API để chấm `model_response` so với `output`, rồi lưu thêm `api_score` và `api_raw_score`.

In [ ]:
import json
import re
import time
import urllib.error
import urllib.request

from tqdm import tqdm

## Load JSON

In [ ]:
file_path = "instruction-data-with-response.json"

with open(file_path, "r", encoding="utf-8") as file:
    test_data = json.load(file)

required_keys = {"instruction", "input", "output", "model_response"}

for i, entry in enumerate(test_data):
    missing_keys = required_keys - set(entry.keys())
    if missing_keys:
        raise KeyError(f"Entry {i} is missing keys: {missing_keys}")

print("Number of entries:", len(test_data))
print("Keys:", list(test_data[0].keys()))
print(test_data[0])

## Format Input

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )

    return instruction_text + input_text

## Gemini API Config

Trên Kaggle, tạo secret:

```text
LABEL: GEMINI_API_KEY
VALUE: key Gemini từ Google AI Studio
```

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")

# Doi model o day neu model hien tai bi qua tai.
GEMINI_MODEL = "gemini-2.0-flash"
GEMINI_URL = (
    f"https://generativelanguage.googleapis.com/v1beta/models/"
    f"{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
)

## Query Gemini

In [ ]:
def extract_text_from_gemini_response(response_json):
    try:
        return response_json["candidates"][0]["content"]["parts"][0]["text"]
    except (KeyError, IndexError, TypeError):
        print("Unexpected Gemini response JSON:")
        print(json.dumps(response_json, indent=2, ensure_ascii=False)[:3000])
        raise


def query_model(prompt, url=GEMINI_URL, max_retries=5):
    data = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0
        }
    }

    payload = json.dumps(data).encode("utf-8")

    for attempt in range(max_retries):
        request = urllib.request.Request(
            url,
            data=payload,
            method="POST"
        )
        request.add_header("Content-Type", "application/json")

        try:
            with urllib.request.urlopen(request) as response:
                response_data = response.read().decode("utf-8")

            response_json = json.loads(response_data)
            return extract_text_from_gemini_response(response_json)

        except urllib.error.HTTPError as e:
            error_text = e.read().decode("utf-8")
            print("HTTP status:", e.code)
            print("API error:", error_text)

            if e.code in [429, 500, 503] and attempt < max_retries - 1:
                wait_time = 10 * (attempt + 1)
                print(f"Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
                continue

            raise

    raise RuntimeError("Gemini API failed after multiple retries.")

## Score Helpers

In [ ]:
def build_eval_prompt(entry, json_key="model_response"):
    return (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry[json_key]}` "
        f"on a scale from 0 to 100, where 100 is the best score. "
        f"Respond with the integer number only."
    )


def extract_integer_score(score_text):
    match = re.search(r"\d+", score_text)

    if match is None:
        raise ValueError(f"Could not convert score: {score_text}")

    score = int(match.group())
    return max(0, min(100, score))

## Test 3 Examples

In [ ]:
for entry in test_data[:3]:
    prompt = build_eval_prompt(entry, json_key="model_response")
    raw_score = query_model(prompt)
    score = extract_integer_score(raw_score)

    print("\nDataset response:")
    print(">>", entry["output"])

    print("\nModel response:")
    print(">>", entry["model_response"])

    print("\nScore:")
    print(">>", score)

    print("\n-------------------------")

## Score Entries

In [ ]:
def generate_model_scores(json_data, json_key="model_response"):
    scores = []

    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = build_eval_prompt(entry, json_key=json_key)
        raw_score = query_model(prompt)

        try:
            score = extract_integer_score(raw_score)
        except ValueError:
            print(f"Could not convert score: {raw_score}")
            continue

        entry["api_score"] = score
        entry["api_raw_score"] = raw_score
        scores.append(score)

    return scores

In [ ]:
# Test nhanh voi 5 mau. Khi on roi, doi test_data[:5] thanh test_data de cham toan bo.
scores = generate_model_scores(test_data[:5], json_key="model_response")

print("Number of scores:", len(scores))
print(f"Average score: {sum(scores) / len(scores):.2f}")

## Save JSON

In [ ]:
output_file = "instruction-data-with-api-scores.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(test_data, file, indent=4, ensure_ascii=False)

print("Saved to:", output_file)